# Python进阶（三）：函数进阶
## 学习目标
- 理解函数作为一等公民的特性
- 掌握高阶函数的使用（map、filter）
- 理解闭包和作用域规则
- 掌握装饰器的使用和编写
- 理解迭代器和生成器的区别

## 一、函数作为一等公民
Python中，函数可以：
1. 赋值给变量
2. 作为函数的参数
3. 作为函数的返回值

In [ ]:
"""函数作为一等公民示例"""
# 1. 函数赋值给变量
def greet(name):
    return f"Hello, {name}!"

say_hello = greet  # 函数赋值给变量
print(say_hello("Alice"))  # Hello, Alice!

# 2. 函数作为参数
def apply_func(func, value):
    return func(value)

print(apply_func(greet, "Bob"))  # Hello, Bob!

# 3. 函数作为返回值（闭包）
def make_multiplier(n):
    def multiplier(x):
        return x * n
    return multiplier

double = make_multiplier(2)
print(double(5))  # 10

## 二、高阶函数
### 1. map() 和 filter()
- `map(func, iterable)`：映射，返回迭代器
- `filter(func, iterable)`：过滤，返回迭代器
- **替代品**：列表生成式（更Pythonic）

In [ ]:
"""高阶函数示例"""
# 方式1：使用map和filter
items1 = list(map(lambda x: x ** 2, filter(lambda x: x % 2, range(1, 11))))
print("方式1：", items1)

# 方式2：使用列表生成式（推荐）
items2 = [x ** 2 for x in range(1, 11) if x % 2]
print("方式2：", items2)

# 两者等价，但方式2更清晰

## 三、闭包和作用域
### 1. Python搜索变量的LEGB顺序
- **L**ocal：局部作用域
- **E**nclosed：嵌套作用域
- **G**lobal：全局作用域
- **B**uilt-in：内置作用域

In [ ]:
"""LEGB规则示例"""
x = 10  # 全局变量

def outer():
    x = 20  # 嵌套作用域变量
    
    def inner():
        x = 30  # 局部变量
        print("inner:", x)  # 30 (Local)
    
    inner()
    print("outer:", x)  # 20 (Enclosed)

outer()
print("global:", x)  # 10 (Global)

"""如果要修改外层变量，需要使用nonlocal或global关键字"""

### 2. global 和 nonlocal 关键字
- `global`：声明使用全局变量
- `nonlocal`：声明使用嵌套作用域的变量

In [ ]:
"""global和nonlocal示例"""
counter = 0  # 全局变量

def increment():
    global counter  # 声明使用全局变量
    counter += 1

increment()
increment()
print("counter:", counter)  # 2


def outer():
    x = 10
    
    def inner():
        nonlocal x  # 声明使用嵌套作用域的变量
        x += 1
    
    inner()
    print("x in outer:", x)  # 11

outer()

## 四、装饰器（Decorator）
### 1. 基本装饰器
装饰器是修改其他函数功能的函数，让你在不修改原函数的情况下，为其添加新功能。

In [ ]:
"""基本装饰器示例"""
import time

def record_time(func):
    """装饰器：记录函数执行时间"""
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        print(f"{func.__name__}: {time.time() - start:.3f}秒")
        return result
    return wrapper

@record_time
def my_sum(n):
    """计算1到n的和"""
    return sum(range(1, n + 1))

# 等价于：my_sum = record_time(my_sum)
print(my_sum(100000))

### 2. 参数化装饰器
可以接收参数的装饰器，更灵活。

In [ ]:
"""参数化装饰器示例"""
from functools import wraps

def record(output=print):
    """参数化装饰器工厂"""
    def decorate(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            start = time.time()
            result = func(*args, **kwargs)
            output(f"{func.__name__}: {time.time() - start:.3f}秒")
            return result
        return wrapper
    return decorate

@record(output=print)
def my_sum(n):
    return sum(range(1, n + 1))

my_sum(100000)

## 五、迭代器和生成器
### 1. 迭代器（Iterator）
- 实现了迭代器协议（`__iter__()` 和 `__next__()`）
- 惰性计算，节省内存
- 一次性，遍历完就耗尽

In [ ]:
"""自定义迭代器示例"""
class Fib:
    """斐波那契数列迭代器"""
    def __init__(self, num):
        self.num = num
        self.a, self.b = 0, 1
        self.idx = 0
    
    def __iter__(self):
        return self
    
    def __next__(self):
        if self.idx < self.num:
            self.a, self.b = self.b, self.a + self.b
            self.idx += 1
            return self.a
        raise StopIteration()

# 测试
fib = Fib(10)
for num in fib:
    print(num, end=" ")

### 2. 生成器（Generator）
- 语法简化的迭代器
- 使用 `yield` 关键字
- 自动实现迭代器协议

In [ ]:
"""生成器示例"""
def fib(num):
    """斐波那契数列生成器"""
    a, b = 0, 1
    for _ in range(num):
        a, b = b, a + b
        yield a

# 测试
for num in fib(10):
    print(num, end=" ")


"""生成器进阶：协程"""
def calc_avg():
    """流式计算平均值"""
    total, counter = 0, 0
    avg_value = None
    while True:
        value = yield avg_value
        total, counter = total + value, counter + 1
        avg_value = total / counter

# 测试
gen = calc_avg()
next(gen)  # 启动生成器
print(gen.send(10))  # 10.0
print(gen.send(20))  # 15.0
print(gen.send(30))  # 20.0

## 六、练习与自测
### 练习题1：编写计时装饰器
**题目**：编写一个装饰器，记录函数执行的时间，并统计函数被调用的次数。
**提示**：使用闭包保存调用次数。

In [ ]:
"""练习题1解答：计时和计数装饰器"""
import time
from functools import wraps

def record_time_and_count(func):
    """装饰器：记录时间和调用次数"""
    count = 0  # 闭包变量
    
    @wraps(func)
    def wrapper(*args, **kwargs):
        nonlocal count
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        count += 1
        print(f"{func.__name__} 第{count}次调用，耗时{elapsed:.3f}秒")
        return result
    
    return wrapper

@record_time_and_count
def my_sum(n):
    return sum(range(1, n + 1))

my_sum(10000)
my_sum(100000)

### 练习题2：自定义迭代器
**题目**：实现一个自定义迭代器类 `Countdown`，从n倒计时到1。

In [ ]:
"""练习题2解答：倒计时迭代器"""
class Countdown:
    """倒计时迭代器"""
    def __init__(self, n):
        self.n = n
    
    def __iter__(self):
        return self
    
    def __next__(self):
        if self.n <= 0:
            raise StopIteration()
        self.n -= 1
        return self.n + 1

# 测试
for num in Countdown(5):
    print(num, end=" ")  # 5 4 3 2 1

## 七、知识总结
### 重点记忆
1. **函数作为一等公民**
   - 可以赋值、作为参数、作为返回值
2. **高阶函数**
   - `map()`：映射
   - `filter()`：过滤
   - 替代品：列表生成式（更推荐）
3. **作用域规则（LEGB）**
   - Local → Enclosed → Global → Built-in
   - `global`：声明全局变量
   - `nonlocal`：声明嵌套作用域变量
4. **装饰器**
   - 基本装饰器：`def decorator(func):`
   - 参数化装饰器：`def decorator(param):`
   - 使用`@wraps(func)`保留原函数元信息
5. **迭代器和生成器**
   - 迭代器：实现`__iter__()`和`__next__()`
   - 生成器：使用`yield`关键字
   - 生成器可以作为协程使用（`send()`方法）